## التضمينات

في مثالنا السابق، تعاملنا مع متجهات حقيبة كلمات عالية الأبعاد بطول `vocab_size`، وكنا نحول صراحةً من تمثيلات الموقع منخفضة الأبعاد إلى تمثيل متفرق بتشفير واحد-ساخن. هذا التمثيل الواحد-ساخن غير فعال من حيث الذاكرة، بالإضافة إلى ذلك، يُعامل كل كلمة بشكل مستقل عن الأخرى، أي أن المتجهات المشفرة بتشفير واحد-ساخن لا تعبر عن أي تشابه معنوي بين الكلمات.

في هذه الوحدة، سنواصل استكشاف مجموعة بيانات **News AG**. للبدء، دعونا نحمل البيانات ونحصل على بعض التعريفات من الدفتر السابق.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## ما هو التضمين؟

فكرة **التضمين** هي تمثيل الكلمات بواسطة متجهات كثيفة منخفضة الأبعاد، والتي تعكس بطريقة ما المعنى الدلالي للكلمة. سنناقش لاحقًا كيفية بناء تضمينات كلمات ذات معنى، ولكن الآن دعونا نفكر في التضمينات فقط كطريقة لتقليل أبعاد متجه الكلمة.

لذلك، تأخذ طبقة التضمين كلمة كمدخل، وتنتج متجهًا مخرجًا بأبعاد محددة `embedding_size`. من ناحية ما، هي مشابهة جدًا لطبقة `Linear`، ولكن بدلاً من أخذ متجه مشفر بطريقة one-hot، ستكون قادرة على أخذ رقم الكلمة كمدخل.

باستخدام طبقة التضمين كطبقة أولى في شبكتنا، يمكننا الانتقال من نموذج حقيبة الكلمات إلى نموذج **حقيبة التضمين**، حيث نحول أولاً كل كلمة في نصنا إلى التضمين الموافق لها، ثم نحسب دالة تجميعية على كل تلك التضمينات، مثل `sum`، `average` أو `max`.

![صورة تُظهر مُصنف التضمين لخمس كلمات في تسلسل.](../../../../../translated_images/ar/embedding-classifier-example.b77f021a7ee67eee.webp)

ستبدأ شبكة التصنيف العصبي لدينا بطبقة التضمين، ثم طبقة التجميع، وأخيرًا مُصنف خطي في الأعلى:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### التعامل مع حجم تسلسل متغير

كنتيجة لهذا التصميم، يجب إنشاء دفعات صغيرة لشبكتنا بطريقة معينة. في الوحدة السابقة، عند استخدام حقيبة الكلمات، كانت جميع موترات BoW في الدفعة الصغيرة ذات حجم متساوٍ `vocab_size`، بغض النظر عن الطول الفعلي لتسلسل النص لدينا. بمجرد الانتقال إلى تضمينات الكلمات، سينتهي بنا الأمر بعدد متغير من الكلمات في كل عينة نص، وعند دمج تلك العينات في دفعات صغيرة يجب علينا تطبيق بعض الحشو.

يمكن القيام بذلك باستخدام نفس التقنية المتمثلة في توفير دالة `collate_fn` لمصدر البيانات:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### تدريب مصنف التضمين

الآن بعد أن قمنا بتعريف مُحمّل البيانات المناسب، يمكننا تدريب النموذج باستخدام وظيفة التدريب التي قمنا بتعريفها في الوحدة السابقة:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **ملاحظة**: نحن نتدرب هنا فقط على 25 ألف سجل (أقل من عصر كامل) من أجل الوقت، ولكن يمكنك الاستمرار في التدريب، وكتابة دالة للتدريب لعدة عصور، والتجربة مع معامل معدل التعلم لتحقيق دقة أعلى. يجب أن تكون قادرًا على الوصول إلى دقة حوالي 90%.


### طبقة EmbeddingBag وتمثيل تسلسل بطول متغير

في البنية السابقة، كنا بحاجة إلى ملء كل التسلسلات لتصبح بنفس الطول من أجل تضمينها في دفعة صغيرة. هذا ليس أكثر الطرق كفاءة لتمثيل التسلسلات ذات الطول المتغير - نهج آخر هو استخدام متجه **الإزاحة**، الذي يحتوي على إزاحات كل التسلسلات المخزنة في متجه كبير واحد.

![صورة تظهر تمثيل تسلسل بالإزاحات](../../../../../translated_images/ar/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **ملاحظة**: في الصورة أعلاه، نعرض تسلسلاً من الأحرف، لكن في مثالنا نعمل مع تسلسلات من الكلمات. ومع ذلك، يبقى المبدأ العام لتمثيل التسلسلات باستخدام متجه الإزاحة كما هو.

للعمل مع تمثيل الإزاحة، نستخدم طبقة [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html). هي مشابهة لـ `Embedding`، لكنها تأخذ متجه المحتوى ومتجه الإزاحة كمدخلات، وتشمل أيضًا طبقة الوسيط، التي يمكن أن تكون `mean`، `sum` أو `max`.

هنا شبكة معدلة تستخدم `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

لتحضير مجموعة البيانات للتدريب، نحتاج إلى توفير دالة تحويل ستقوم بتحضير متجه الإزاحة:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

لاحظ أنه، على عكس جميع الأمثلة السابقة، شبكتنا الآن تقبل معاملين: متجه البيانات ومتجه الإزاحة، اللذان لهما أحجام مختلفة. وبالمثل، يوفر لنا محمل البيانات الخاص بنا 3 قيم بدلاً من 2: يتم توفير كل من متجهات النص والإزاحة كميزات. لذلك، نحتاج إلى تعديل طفيف في وظيفة التدريب لدينا للتعامل مع ذلك:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## التضمينات الدلالية: Word2Vec

في مثالنا السابق، تعلم طبقة التضمين في النموذج تعيين الكلمات إلى تمثيل متجه، ومع ذلك لم يكن لهذا التمثيل معنى دلالي كبير. سيكون من الجميل تعلم تمثيل متجهة بحيث تكون الكلمات المتشابهة أو المرادفات مطابقة لمتجهات تكون قريبة من بعضها البعض من حيث بعض مسافات المتجهات (مثل المسافة الإقليدية).

للقيام بذلك، نحتاج إلى تدريب نموذج التضمين مسبقًا على مجموعة كبيرة من النصوص بطريقة محددة. واحدة من الطرق الأولى لتدريب التضمينات الدلالية تسمى [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). وهي تعتمد على بنائين رئيسيين يُستخدمان لإنتاج تمثيل موزع للكلمات:

 - **حقيبة الكلمات المستمرة** (CBoW) — في هذا البناء، ندرب النموذج للتنبؤ بكلمة من السياق المحيط بها. بالنظر إلى ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$، فإن هدف النموذج هو التنبؤ بـ $W_0$ من $(W_{-2},W_{-1},W_1,W_2)$.
 - **السكب-غرام المستمر** هو عكس CBoW. يستخدم النموذج نافذة من كلمات السياق المحيطة للتنبؤ بالكلمة الحالية.

CBoW أسرع، بينما سكب-غرام أبطأ، لكنه يقوم بتمثيل أفضل للكلمات النادرة.

![صورة تُظهر خوارزميات CBoW وSkip-Gram لتحويل الكلمات إلى متجهات.](../../../../../translated_images/ar/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

لتجربة تضمين word2vec المدرب مسبقًا على مجموعة بيانات أخبار Google، يمكننا استخدام مكتبة **gensim**. أدناه نجد الكلمات الأكثر تشابهًا مع 'neural'

> **ملاحظة:** عند إنشاء متجهات الكلمات لأول مرة، قد يستغرق تنزيلها بعض الوقت!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


يمكننا أيضًا حساب تضمينات المتجهات من الكلمة، لاستخدامها في تدريب نموذج التصنيف (نعرض فقط أول 20 مكونًا من المتجه للوضوح):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

الشيء الرائع في التضمينات الدلالية هو أنه يمكنك التلاعب في ترميز المتجهات لتغيير الدلالة. على سبيل المثال، يمكننا أن نطلب إيجاد كلمة، يكون تمثيلها المتجهي قريبًا قدر الإمكان من الكلمات *king* و *woman*، وبعيدة قدر الإمكان عن كلمة *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

كل من CBoW و Skip-Grams هما تمثيلات مضمنة "تنبؤية"، بمعنى أنهما يأخذان السياقات المحلية فقط في الاعتبار. لا يستفيد Word2Vec من السياق العالمي.

**FastText**، يبني على Word2Vec عن طريق تعلم تمثيلات متجهة لكل كلمة وتضمين ن-جرام (تسلسلات الحروف) داخل كل كلمة. ثم يتم حساب متوسط قيم التمثيلات في متجه واحد في كل خطوة تدريب. رغم أن هذا يضيف الكثير من العمليات الحسابية الإضافية على مرحلة ما قبل التدريب، فإنه يتيح للتمثيلات المضمنة تشفير معلومات تحت الكلمة.

طريقة أخرى، **GloVe**، تعتمد على فكرة مصفوفة التواجد المشترك، وتستخدم الطرق العصبية لتحليل مصفوفة التواجد المشترك إلى متجهات كلمات أكثر تعبيرًا وغير خطية.

يمكنك تجربة المثال بتغيير التمثيلات المضمنة إلى FastText و GloVe، حيث يدعم gensim العديد من نماذج التمثيل المضمنة المختلفة.


## استخدام التمثيلات المسبقة التدريب في بايتورتش

يمكننا تعديل المثال أعلاه لملء مصفوفة طبقة التمثيل الخاصة بنا بتمثيلات دلالية مسبقة التدريب، مثل Word2Vec. نحتاج إلى أخذ في الاعتبار أن مفردات التمثيل المسبق التدريب ومجموعتنا النصية من المحتمل ألا تتطابق، لذلك سنقوم بتهيئة الأوزان للكلمات المفقودة بقيم عشوائية:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


الآن دعونا ندرب نموذجنا. لاحظ أن الوقت اللازم لتدريب النموذج أكبر بكثير من المثال السابق، وذلك بسبب حجم طبقة التضمين الأكبر، وبالتالي عدد المعاملات الأعلى بكثير. أيضًا، بسبب هذا، قد نحتاج إلى تدريب نموذجنا على مزيد من الأمثلة إذا أردنا تجنب الإفراط في التخصيص.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

في حالتنا، لا نرى زيادة كبيرة في الدقة، والتي من المحتمل أن تكون بسبب اختلاف كبير في المصطلحات. 
للتغلب على مشكلة اختلاف المصطلحات، يمكننا استخدام أحد الحلول التالية:
* إعادة تدريب نموذج word2vec على قاموسنا الخاص
* تحميل مجموعة البيانات الخاصة بنا بالقاموس من نموذج word2vec المدرب مسبقًا. يمكن تحديد القاموس المستخدم لتحميل مجموعة البيانات أثناء التحميل.

يبدو أن النهج الأخير أسهل، خاصة لأن إطار عمل PyTorch `torchtext` يحتوي على دعم مدمج للتضمينات. يمكننا، على سبيل المثال، إنشاء قاموس قائم على GloVe بالطريقة التالية:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


تحتوي المفردات المحملة على العمليات الأساسية التالية:
* `vocab.stoi` القاموس يسمح لنا بتحويل الكلمة إلى مؤشرها في القاموس
* `vocab.itos` يقوم بالعكس - يحول الرقم إلى كلمة
* `vocab.vectors` هو مصفوفة متجهات التضمين، لذا للحصول على تضمين كلمة `s` نحتاج إلى استخدام `vocab.vectors[vocab.stoi[s]]`

هنا مثال على التلاعب بالتضمينات لتوضيح المعادلة **kind-man+woman = queen** (اضطررت إلى تعديل المعامل قليلاً لجعلها تعمل):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

لتدريب المُصنِّف باستخدام تلك التضمينات، نحتاج أولاً إلى ترميز مجموعة البيانات الخاصة بنا باستخدام مفردات GloVe:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

كما رأينا أعلاه، يتم تخزين جميع تضمينات المتجهات في مصفوفة `vocab.vectors`. هذا يجعل من السهل جدًا تحميل تلك الأوزان إلى أوزان طبقة التضمين باستخدام نسخ بسيط:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

الآن دعونا ندرب نموذجنا ونرى إذا كنا سنحصل على نتائج أفضل:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

أحد الأسباب التي تجعلنا لا نرى زيادة كبيرة في الدقة هو حقيقة أن بعض الكلمات من مجموعة بياناتنا مفقودة في مفردات GloVe المدربة مسبقًا، وبالتالي يتم تجاهلها فعليًا. للتغلب على هذه الحقيقة، يمكننا تدريب تمثيلات خاصة بنا على مجموعة بياناتنا.


## التضمينات السياقية

إحدى القيود الرئيسية لتمثيلات التضمين المدربة مسبقًا التقليدية مثل Word2Vec هي مشكلة تحديد معنى الكلمة. بينما يمكن للتضمينات المدربة مسبقًا أن تلتقط بعض معاني الكلمات في السياق، يتم ترميز كل معنى ممكن للكلمة في نفس التضمين. يمكن أن يسبب هذا مشاكل في النماذج اللاحقة، حيث أن العديد من الكلمات مثل كلمة "play" لها معانٍ مختلفة حسب السياق الذي تُستخدم فيه.

على سبيل المثال، كلمة "play" في هاتين الجملتين المختلفتين لها معنى مختلف تمامًا:
- ذهبت إلى **مسرحية** في المسرح.
- جون يريد **اللعب** مع أصدقائه.

التضمينات المدربة مسبقًا أعلاه تمثل كلا المعنيين لكلمة "play" في نفس التضمين. للتغلب على هذا القيد، نحتاج إلى بناء تضمينات تعتمد على **نموذج اللغة**، الذي يتم تدريبه على مجموعة كبيرة من النصوص، ويعرف كيفية تركيب الكلمات معًا في سياقات مختلفة. مناقشة التضمينات السياقية خارج نطاق هذا الدرس، لكننا سنعود إليها عند التحدث عن نماذج اللغة في الوحدة التالية.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
